# Séance 1 — Environnement de travail et NumPy
²**Durée pratique : 2 h** &nbsp;·&nbsp; Ateliers 1.1 à 1.3

## Ce que vous saurez faire à la fin

- créer un environnement virtuel reproductible et figer ses dépendances ;
- manipuler des tableaux NumPy : slicing, masques booléens, statistiques par axe ;
- mesurer et expliquer l'écart de performance entre une boucle Python et une opération vectorisée.

> **Convention du module.** Le dossier `data/raw/` est en lecture seule. Toute transformation
> produit un nouveau fichier dans `data/processed/`. Aucune donnée brute n'est jamais écrasée.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

## Préparation de la session

In [13]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  capteurs.csv
  clients.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


A---
## Atelier 1.1 — Environnement et projet (30 min)

Cet atelier se fait **dans un terminal**, pas dans le notebook. Les commandes sont
données ci-dessous ; la cellule suivante sert uniquement à vérifier le résultat.

```bash
# 1. Créer et activer l'environnement
python -m venv .venv
source .venv\Scripts\activate

# 2. Installer les dépendances
pip install pandas numpy scikit-learn matplotlib seaborn pyarrow jupyterlab

# 3. Figer l'état exact de l'environnement
pip freeze > requirements.txt

# 4. Versionner le projet
git init
printf '.venv/\ndata/raw/\n__pycache__/\n.ipynb_checkpoints/\n' > .gitignore
git add . && git commit -m 'Initialisation du projet PMD'
```

**Pourquoi `data/raw/` dans le `.gitignore` ?** Les données brutes peuvent être
volumineuses et ne changent jamais. On versionne le *code qui les transforme*, pas
les données elles-mêmes. Le `README` doit indiquer où les récupérer.

In [2]:
# Vérification : l'interpréteur utilisé est-il bien celui de l'environnement virtuel ?
import sys

print('Interpréteur :', sys.executable)
print('Version      :', sys.version.split()[0])

for package_name in ['numpy', 'pandas', 'sklearn', 'matplotlib']:
    try:
        module = __import__(package_name)
        print(f'{package_name:<12} {getattr(module, "__version__", "?")}')
    except ImportError:
        print(f'{package_name:<12} ABSENT')



Interpréteur : C:\Users\flori\Downloads\Préparation et manipulation de données\.venv\Scripts\python.exe
Version      : 3.12.10
numpy        2.5.3
pandas       2.2.3
sklearn      ABSENT
matplotlib   3.11.2


In [3]:
!git add .
!git commit -m "Initialisation du projet PMD"


[master 6746462] Initialisation du projet PMD
 14 files changed, 61871 insertions(+), 215 deletions(-)
 create mode 100644 03_nettoyage_qualite.ipynb
 create mode 100644 capteurs.csv
 create mode 100644 clients.csv
 create mode 100644 generate_data.py
 create mode 100644 magasins.csv
 create mode 100644 produits.csv
 create mode 100644 ventes_brutes.csv
 create mode 100644 ventes_extrait.json
 create mode 100644 ventes_extrait.xlsx
 create mode 100644 word TD/support_copilot_python_data.docx
 create mode 100644 word TD/support_copilot_python_data.md


**À vérifier avant de continuer.** Le chemin de l'interpréteur doit contenir `.venv`.
S'il pointe vers le Python système, le noyau du notebook n'est pas celui de votre
environnement : installez `ipykernel` dans le venv et sélectionnez le bon noyau.

---
## Atelier 1.2 — NumPy : le tableau comme unité de calcul (50 min)

### Partie guidée

Un `ndarray` a un **type unique** pour toutes ses valeurs. C'est ce qui permet à NumPy de
stocker les données de façon contiguë en mémoire et d'appliquer une opération à l'ensemble
du tableau sans boucle Python.

In [4]:
import numpy as np
rng = np.random.default_rng(42)

# 12 relevés de température pour 5 capteurs : matrice 5 lignes x 12 colonnes
temperatures = np.round(rng.normal(loc=14, scale=6, size=(5, 12)), 1)

print('forme   :', temperatures.shape)
print('type    :', temperatures.dtype)
print('mémoire :', temperatures.nbytes, 'octets')
temperatures

forme   : (5, 12)
type    : float64
mémoire : 480 octets


array([[15.8,  7.8, 18.5, 19.6,  2.3,  6.2, 14.8, 12.1, 13.9,  8.9, 19.3,
        18.7],
       [14.4, 20.8, 16.8,  8.8, 16.2,  8.2, 19.3, 13.7, 12.9,  9.9, 21.3,
        13.1],
       [11.4, 11.9, 17.2, 16.2, 16.5, 16.6, 26.8, 11.6, 10.9,  9.1, 17.7,
        20.8],
       [13.3,  9. ,  9.1, 17.9, 18.5, 17.3, 10. , 15.4, 14.7, 15.3, 19.2,
        15.3],
       [18.1, 14.4, 15.7, 17.8,  5.3, 12.1, 11.2, 10.2, 12.3, 23. ,  8.8,
        19.8]])

In [5]:
# Statistiques par axe : axis=0 parcourt les lignes, axis=1 parcourt les colonnes
print('Moyenne par mois (sur les 5 capteurs) :', np.round(temperatures.mean(axis=0), 2))
print('Moyenne par capteur (sur les 12 mois) :', np.round(temperatures.mean(axis=1), 2))
print('Moyenne globale                       :', round(temperatures.mean(), 2))

Moyenne par mois (sur les 5 capteurs) : [14.6  12.78 15.46 16.06 11.76 12.08 16.42 12.6  12.94 13.24 17.26 17.54]
Moyenne par capteur (sur les 12 mois) : [13.16 14.62 15.56 14.58 14.06]
Moyenne globale                       : 14.4


**L'axe est la dimension qui disparaît.** `axis=0` agrège sur les lignes et laisse
12 valeurs, une par colonne. C'est la source d'erreur numéro un chez les débutants.

In [6]:
# Masque booléen : un tableau de True/False de même forme que l'original
is_freezing = temperatures < 0

print('Nombre de relevés sous zéro :', is_freezing.sum())
print('Capteurs concernés          :', np.where(is_freezing.any(axis=1))[0])

# Le masque sert à lire...
print('Valeurs négatives           :', temperatures[is_freezing])

# ... et à écrire
corrected = temperatures.copy()
corrected[is_freezing] = 0
print('Minimum après correction    :', corrected.min())

Nombre de relevés sous zéro : 0
Capteurs concernés          : []
Valeurs négatives           : []
Minimum après correction    : 2.3


### Partie autonome

Complétez les cellules suivantes. Chaque cellule se termine par un `assert` :
s'il passe sans message, votre réponse est correcte.

In [7]:
# Q1. Construire un tableau des écarts de chaque relevé à la moyenne de SON capteur.
#     Attendu : forme (5, 12), et la moyenne de chaque ligne doit valoir 0.
#     Indice : temperatures.mean(axis=1) a la forme (5,) ; il faut la remettre en (5, 1)
#     pour que le broadcasting aligne les lignes. Voir .reshape(-1, 1) ou [:, None].

# Moyenne par capteur, remise en colonne
means = temperatures.mean(axis=1).reshape(-1, 1)

# Écarts à la moyenne
deviations = temperatures - means

# Vérifications
assert deviations.shape == (5, 12), 'forme incorrecte'
assert np.allclose(deviations.mean(axis=1), 0), 'les lignes ne sont pas centrées'

print('OK — écart type des écarts :', round(deviations.std(), 2))


OK — écart type des écarts : 4.61


In [8]:
# Q2. Compter, pour chaque capteur, le nombre de mois où la température dépasse
#     la moyenne globale de tout le tableau.
#     Attendu : un tableau de 5 entiers.

months_above_mean = ... # TODO

global_mean = temperatures.mean()
mask = temperatures > global_mean
months_above_mean = mask.sum(axis=1)

assert months_above_mean.shape == (5,), 'un compte par capteur est attendu'
assert months_above_mean.sum() == (temperatures > temperatures.mean()).sum()
print('OK —', months_above_mean)

OK — [6 6 7 8 6]


In [9]:
# Q3. Remplacer les valeurs aberrantes par la médiane du tableau.
#     Est aberrante toute valeur à plus de 2 écarts types de la moyenne globale.
#     Travaillez sur une COPIE : le tableau d'origine ne doit pas changer.

cleaned = temperatures.copy()

mean = temperatures.mean()
std = temperatures.std()
median = np.median(temperatures)

mask = np.abs(temperatures - mean) > 2 * std
cleaned[mask] = median

assert cleaned is not temperatures, 'vous avez modifié le tableau original'
assert cleaned.shape == temperatures.shape
threshold = 2 * temperatures.std()
assert np.abs(cleaned - temperatures.mean()).max() <= threshold + 1e-9
print('OK — valeurs remplacées :', int((cleaned != temperatures).sum()))

OK — valeurs remplacées : 2


---
## Atelier 1.3 — Pourquoi vectoriser (40 min)

L'argument est souvent présenté comme une question de style. C'est une question d'ordre
de grandeur. Mesurez-le vous-même.

In [10]:
import time


def benchmark(func, *args, n_repeats=3):
    """Renvoie (meilleur temps en secondes, résultat de la fonction)."""
    durations = []
    for _ in range(n_repeats):
        start = time.perf_counter()
        result = func(*args)
        durations.append(time.perf_counter() - start)
    return min(durations), result


values = rng.normal(size=2_000_000)
print('Tableau de', f'{values.size:,}'.replace(',', ' '), 'valeurs')

Tableau de 2 000 000 valeurs


In [11]:
def sum_squares_loop(x):
    total = 0.0
    for value in x:
        total += value * value
    return total


def sum_squares_comprehension(x):
    return sum(value * value for value in x)


def sum_squares_numpy(x):
    return np.sum(x ** 2)


timings = {}
for label, func in [('boucle for', sum_squares_loop),
                    ('compréhension', sum_squares_comprehension),
                    ('numpy vectorisé', sum_squares_numpy)]:
    duration, result = benchmark(func, values)
    timings[label] = duration
    print(f'{label:<18} {duration:7.4f} s   (résultat {result:.2f})')

baseline = timings['numpy vectorisé']
print()
for label, duration in timings.items():
    print(f'{label:<18} x{duration / baseline:6.1f} par rapport à NumPy')

boucle for          0.1772 s   (résultat 1999446.99)
compréhension       0.1998 s   (résultat 1999446.99)
numpy vectorisé     0.0074 s   (résultat 1999446.99)

boucle for         x  23.8 par rapport à NumPy
compréhension      x  26.9 par rapport à NumPy
numpy vectorisé    x   1.0 par rapport à NumPy


In [12]:
# Q4. Même comparaison pour un filtrage : compter les valeurs supérieures à 1,5.
#     Écrivez les deux versions et comparez.

def count_above_loop(x, threshold=1.5):
    count = 0
    for value in x:
        if value > threshold:
            count += 1
    return count


import numpy as np

def count_above_numpy(x, threshold=1.5):
    return np.sum(x > threshold)


loop_duration, loop_result = benchmark(count_above_loop, values)
numpy_duration, numpy_result = benchmark(count_above_numpy, values)

assert loop_result == numpy_result, 'les deux versions ne donnent pas le même résultat'
print(f'boucle {loop_duration:.4f} s | numpy {numpy_duration:.4f} s '
      f'| facteur x{loop_duration / numpy_duration:.0f}')

boucle 0.0807 s | numpy 0.0021 s | facteur x39


### À retenir

L'écart typique est d'un facteur 30 à 100. Il ne vient pas de la vitesse du calcul lui-même
mais du coût de l'interprétation : chaque tour de boucle Python crée des objets, vérifie des
types et appelle des méthodes. NumPy délègue la boucle à du code compilé qui travaille
directement sur un bloc mémoire homogène.

**Conséquence pratique pour le reste du module :** dès que vous écrivez `for` sur les lignes
d'un DataFrame, arrêtez-vous et cherchez l'opération vectorisée équivalente.

---
## Livrable de la séance

Un dépôt Git contenant :

- `requirements.txt` généré par `pip freeze` ;
- `.gitignore` excluant `.venv/` et `data/raw/` ;
- ce notebook exécuté, avec les quatre questions complétées ;
- un `README.md` de dix lignes décrivant le projet et la procédure d'installation.